# Analyse des points XOR — Réseau neuronal analogique
Lecture du fichier de mesures, affichage des fréquences par point (Vx, Vy) et classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

## 1. Lecture du fichier CSV

In [ ]:
CSV_PATH = "2442xor2.csv"  # adapter le chemin si nécessaire

raw = pd.read_csv(CSV_PATH, header=None)

# Extraire les colonnes de données (à partir de la colonne index 3)
data_cols = raw.iloc[0, 3:].values   # noms des colonnes C0_0 … C0_24

def parse_row(label):
    """Retourne le vecteur numérique de la ligne dont la 2e colonne == label."""
    mask = raw.iloc[:, 1] == label
    row  = raw.loc[mask].iloc[0, 3:].values
    return pd.to_numeric(row, errors='coerce').astype(float)

Vex_X   = parse_row('Vex_X')     # tensions Vin1 [V]
Vex_Y   = parse_row('Vex_Y')     # tensions Vin2 [V]
F_class1 = parse_row('F_class1') # fréquence sortie classe 1 [Hz]
F_class0 = parse_row('F_class0') # fréquence sortie classe 0 [Hz]

n_points = len(Vex_X)
print(f"✓ {n_points} points chargés.")
print(f"  Vex_X  (mV) : {np.unique(Vex_X*1e3)}")
print(f"  Vex_Y  (mV) : {np.unique(Vex_Y*1e3)}")

## 2. Classification point par point
- **Classe 1** si F_class1 > F_class0  
- **Classe 0** si F_class0 > F_class1  
- **Indéterminé** si F_class0 == F_class1

In [ ]:
labels = np.where( F_class1 > F_class0, 1,
         np.where(F_class0 > F_class1, 0, -1))  # -1 = indéterminé

marge  = (F_class1 - F_class0) * 1e-3  # en kHz, signée

df = pd.DataFrame({
    'Point'    : [f'C0_{i}' for i in range(n_points)],
    'Vx (mV)'  : (Vex_X  * 1e3).round(1),
    'Vy (mV)'  : (Vex_Y  * 1e3).round(1),
    'F_class1 (kHz)': (F_class1 * 1e-3).round(2),
    'F_class0 (kHz)': (F_class0 * 1e-3).round(2),
    'Marge (kHz)'   : marge.round(3),
    'Classe prédite': labels
})

print(f"\nClasse 1 : {np.sum(labels==1)} points")
print(f"Classe 0 : {np.sum(labels==0)} points")
if np.any(labels==-1):
    print(f"Indéterminé : {np.sum(labels==-1)} points")

# Affichage console formaté (sans jinja2)
sym = {1: '[C1]', 0: '[C0]', -1: '[??]'}
header = f"{'Point':<8} {'Vx(mV)':>8} {'Vy(mV)':>8} {'F1(kHz)':>10} {'F0(kHz)':>10} {'Marge(kHz)':>12} {'Classe':>7}"
print('\n' + header)
print('-' * len(header))
for _, row in df.iterrows():
    tag = sym[row['Classe prédite']]
    print(f"{row['Point']:<8} {row['Vx (mV)']:>8.1f} {row['Vy (mV)']:>8.1f} "
          f"{row['F_class1 (kHz)']:>10.2f} {row['F_class0 (kHz)']:>10.2f} "
          f"{row['Marge (kHz)']:>+12.3f} {tag:>7}")

## 3. Carte de décision dans l'espace (Vx, Vy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Carte de décision — réseau XOR analogique", fontsize=13, fontweight='bold')

# ── Couleurs des classes ──────────────────────────────────────
color_map = {0: '#EF9F27', 1: '#378ADD', -1: '#E24B4A'}
label_map = {0: 'Classe 0  (F0 > F1)', 1: 'Classe 1  (F1 > F0)', -1: 'Indéterminé'}
marker_map = {0: 's', 1: 'o', -1: 'D'}

# ── Figure gauche : classes ───────────────────────────────────
ax = axes[0]
for cls in np.unique(labels):
    mask = labels == cls
    ax.scatter(
        Vex_X[mask] * 1e3, Vex_Y[mask] * 1e3,
        s=220, c=color_map[cls], marker=marker_map[cls],
        edgecolors='white', linewidths=1.2, zorder=3,
        label=label_map[cls]
    )
    # Annoter chaque point
    for xi, yi, lbl in zip(Vex_X[mask]*1e3, Vex_Y[mask]*1e3, labels[mask]):
        ax.text(xi, yi + 6, str(lbl), ha='center', va='bottom', fontsize=8,
                color=color_map[lbl], fontweight='bold')

ax.set_xlabel("Vx (mV)")
ax.set_ylabel("Vy (mV)")
ax.set_title("Classe prédite par point")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
ax.set_aspect('equal')

# ── Figure droite : marge de confiance ───────────────────────
ax2 = axes[1]
sc = ax2.scatter(
    Vex_X * 1e3, Vex_Y * 1e3,
    c=marge, cmap='RdBu', s=220,
    edgecolors='white', linewidths=1.2, zorder=3,
    vmin=-np.max(np.abs(marge)), vmax=np.max(np.abs(marge))
)
plt.colorbar(sc, ax=ax2, label="Marge F1−F0 (kHz)")
# Annoter la valeur de marge
for xi, yi, m in zip(Vex_X*1e3, Vex_Y*1e3, marge):
    ax2.text(xi, yi + 6, f"{m:+.2f}", ha='center', va='bottom', fontsize=7)

ax2.set_xlabel("Vx (mV)")
ax2.set_ylabel("Vy (mV)")
ax2.set_title("Marge de confiance F1 − F0 (kHz)")
ax2.grid(True, alpha=0.25)
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 4. Grille de décision (format tableau 2D)
Affichage matriciel : Vx en colonnes, Vy en lignes.

In [ ]:
Vx_unique = np.sort(np.unique(Vex_X))
Vy_unique = np.sort(np.unique(Vex_Y))[::-1]  # Vy décroissant pour que +100 soit en haut

# ── Grille des classes ────────────────────────────────────────
grid_cls   = np.full((len(Vy_unique), len(Vx_unique)), np.nan)
grid_f1    = np.full_like(grid_cls, np.nan)
grid_f0    = np.full_like(grid_cls, np.nan)
grid_marge = np.full_like(grid_cls, np.nan)

for k in range(n_points):
    ix = np.where(Vx_unique == Vex_X[k])[0][0]
    iy = np.where(Vy_unique == Vex_Y[k])[0][0]
    grid_cls[iy, ix]   = labels[k]
    grid_f1[iy, ix]    = F_class1[k] * 1e-3
    grid_f0[iy, ix]    = F_class0[k] * 1e-3
    grid_marge[iy, ix] = marge[k]

# ── Affichage matplotlib de la grille ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
cmap_grid = ListedColormap(['#EF9F27', '#378ADD'])
im = ax.imshow(grid_cls, cmap=cmap_grid, vmin=-0.5, vmax=1.5,
               aspect='auto', origin='upper')

ax.set_xticks(range(len(Vx_unique)))
ax.set_xticklabels([f"{v*1e3:.0f}" for v in Vx_unique])
ax.set_yticks(range(len(Vy_unique)))
ax.set_yticklabels([f"{v*1e3:.0f}" for v in Vy_unique])
ax.set_xlabel("Vx (mV)")
ax.set_ylabel("Vy (mV)")
ax.set_title("Grille de décision 2D")

# Annoter chaque cellule : classe + marge
for iy in range(len(Vy_unique)):
    for ix in range(len(Vx_unique)):
        cls = int(grid_cls[iy, ix])
        m   = grid_marge[iy, ix]
        txt_color = 'white' if cls == 1 else 'black'
        ax.text(ix, iy - 0.15, f"Classe {cls}",
                ha='center', va='center', fontsize=9,
                fontweight='bold', color=txt_color)
        ax.text(ix, iy + 0.25, f"{m:+.2f} kHz",
                ha='center', va='center', fontsize=7.5, color=txt_color)

patches = [
    mpatches.Patch(color='#EF9F27', label='Classe 0  (F0 > F1)'),
    mpatches.Patch(color='#378ADD', label='Classe 1  (F1 > F0)')
]
ax.legend(handles=patches, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

# ── Tableaux pandas ───────────────────────────────────────────
col_labels = [f"Vx={v*1e3:.0f}mV" for v in Vx_unique]
row_labels = [f"Vy={v*1e3:.0f}mV" for v in Vy_unique]

print("\n── Tableau des classes ──")
display(pd.DataFrame(grid_cls.astype(int), index=row_labels, columns=col_labels))

print("\n── Tableau F_class1 (kHz) ──")
display(pd.DataFrame(grid_f1.round(2), index=row_labels, columns=col_labels))

print("\n── Tableau F_class0 (kHz) ──")
display(pd.DataFrame(grid_f0.round(2), index=row_labels, columns=col_labels))

print("\n── Tableau marge F1−F0 (kHz) ──")
display(pd.DataFrame(grid_marge.round(3), index=row_labels, columns=col_labels))

## 5. Résumé statistique

In [ ]:
print("=" * 50)
print(f"  Total de points     : {n_points}")
print(f"  Classe 1 (F1 > F0)  : {np.sum(labels==1):2d} points ({100*np.mean(labels==1):.1f}%)")
print(f"  Classe 0 (F0 > F1)  : {np.sum(labels==0):2d} points ({100*np.mean(labels==0):.1f}%)")
if np.any(labels==-1):
    print(f"  Indéterminé         : {np.sum(labels==-1):2d} points")
print("=" * 50)
print(f"  Marge moy. Classe 1 : {marge[labels==1].mean():.3f} kHz" if np.any(labels==1) else "")
print(f"  Marge moy. Classe 0 : {marge[labels==0].mean():.3f} kHz" if np.any(labels==0) else "")
print(f"  Marge max (abs)     : {np.max(np.abs(marge)):.3f} kHz")
print(f"  Marge min (abs)     : {np.min(np.abs(marge)):.3f} kHz")
print("=" * 50)

# Points les plus ambigus (marge faible)
idx_ambig = np.argsort(np.abs(marge))[:3]
print("\n── Points les plus ambigus (marge |F1−F0| la plus faible) ──")
for i in idx_ambig:
    print(f"  C0_{i:02d} : Vx={Vex_X[i]*1e3:.0f}mV, Vy={Vex_Y[i]*1e3:.0f}mV, "
          f"F1={F_class1[i]*1e-3:.2f}kHz, F0={F_class0[i]*1e-3:.2f}kHz, "
          f"marge={marge[i]:+.3f}kHz → Classe {labels[i]}")